# Generative AI Lab Programs — Labs 16 to 25
### Embeddings, Semantic Search, Vector Databases, RAG & LangChain Chatbots

Covers:
- **Lab 16:** Text Embeddings & Semantic Similarity Search
- **Lab 17:** Semantic Search System (Cosine Similarity between Document & Query)
- **Lab 18:** Vector Database with FAISS
- **Lab 19:** Document Storage & Top-k Retrieval with ChromaDB
- **Lab 20:** Document Question Answering using RAG
- **Lab 21:** End-to-end RAG Pipeline (Load → Chunk → Embed → Retrieve → Generate)
- **Lab 22:** Domain-specific Chatbot using LangChain + Vector DB
- **Lab 23:** Context-aware Chatbot using LangChain + Retrieval + LLM (with memory)
- **Lab 24:** Simple AI Assistant answering from an external document
- **Lab 25:** Multi-document AI Assistant with Question Answering

**How to use:** Run the Setup cells first, then each Lab's cells top to bottom.
The LLM used throughout is **Groq** (free & fast). Embeddings are generated **locally**
using `sentence-transformers`, so no separate embedding API key is needed.


## Setup — Install libraries and configure clients
Get a free Groq API key from https://console.groq.com/keys

In [ ]:
# Install all required libraries
!pip install groq sentence-transformers faiss-cpu chromadb -q
!pip install langchain langchain-community langchain-groq langchain-huggingface -q

In [ ]:
from groq import Groq
from getpass import getpass

GROQ_API_KEY = getpass("Enter your Groq API Key: ")

client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.3-70b-versatile"

def ask_llm(prompt, max_tokens=600, temperature=0.5):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print("Groq LLM client ready!")

In [ ]:
# Local embedding model (no API key required, runs on CPU)
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts):
    """Takes a list of strings, returns a numpy array of embedding vectors."""
    return embed_model.encode(texts, convert_to_numpy=True)

print("Embedding model ready! Embedding size:", embed_model.get_sentence_embedding_dimension())

In [ ]:
# A small sample document corpus reused across the search labs (16-19)
corpus = [
    "Cloud computing lets businesses access servers and storage over the internet.",
    "Machine learning models improve automatically through experience and data.",
    "Cybersecurity protects systems and networks from digital attacks.",
    "Blockchain is a distributed ledger technology used in cryptocurrencies.",
    "Natural language processing enables computers to understand human language.",
    "Data visualization turns raw data into charts and graphs for insight.",
    "The Internet of Things connects everyday devices to the internet.",
    "DevOps combines software development and IT operations for faster delivery.",
    "Computer vision allows machines to interpret and understand images.",
    "Big data refers to extremely large datasets analyzed to reveal patterns.",
]
print(f"Corpus loaded with {len(corpus)} documents.")

---
# Lab 16: Generate Text Embeddings and Perform Semantic Similarity Search

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Generate embeddings for the whole corpus
corpus_embeddings = embed_texts(corpus)
print("Corpus embeddings shape:", corpus_embeddings.shape)

In [ ]:
query = "How do companies keep their networks safe from hackers?"
query_embedding = embed_texts([query])

# Compute cosine similarity between the query and every document
similarities = cosine_similarity(query_embedding, corpus_embeddings)[0]

# Rank documents by similarity score
ranked = sorted(zip(corpus, similarities), key=lambda x: x[1], reverse=True)

print(f"Query: {query}\n")
print("Ranked results (most similar first):\n")
for doc, score in ranked:
    print(f"{score:.4f}  -  {doc}")

---
# Lab 17: Semantic Search System using Cosine Similarity
Wrapping the same idea into a reusable `semantic_search()` function that returns the
top-k most relevant documents for any query.

In [ ]:
def semantic_search(query, documents, top_k=3):
    doc_embeddings = embed_texts(documents)
    query_embedding = embed_texts([query])
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]

    results = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return results

# Try it out with a couple of different queries
for q in ["What technology understands human language?", "How do I visualize my data?"]:
    print(f"Query: {q}")
    for doc, score in semantic_search(q, corpus, top_k=3):
        print(f"   {score:.4f}  -  {doc}")
    print()

---
# Lab 18: Build a Vector Database using FAISS
Storing document embeddings in a FAISS index and performing similarity-based retrieval.

In [ ]:
import faiss

# Build a FAISS index (Inner Product = cosine similarity, since we normalize vectors)
dimension = corpus_embeddings.shape[1]

# Normalize embeddings so Inner Product behaves like Cosine Similarity
faiss.normalize_L2(corpus_embeddings)

index = faiss.IndexFlatIP(dimension)
index.add(corpus_embeddings)

print("FAISS index built with", index.ntotal, "vectors.")

In [ ]:
def faiss_search(query, top_k=3):
    query_vec = embed_texts([query])
    faiss.normalize_L2(query_vec)

    scores, indices = index.search(query_vec, top_k)

    print(f"Query: {query}\n")
    for score, idx in zip(scores[0], indices[0]):
        print(f"{score:.4f}  -  {corpus[idx]}")

faiss_search("Tell me about connected smart devices", top_k=3)

---
# Lab 19: Document Storage and Top-k Retrieval using ChromaDB

In [ ]:
import chromadb

chroma_client = chromadb.Client()

# Create (or reset) a collection to store our documents
collection = chroma_client.get_or_create_collection(name="tech_docs")

# Add documents along with unique IDs. Chroma will embed them internally,
# but we can also pass our own precomputed embeddings for consistency.
doc_ids = [f"doc_{i}" for i in range(len(corpus))]

collection.add(
    documents=corpus,
    embeddings=corpus_embeddings.tolist(),
    ids=doc_ids
)

print("Documents stored in ChromaDB collection. Count:", collection.count())

In [ ]:
def chroma_top_k_search(query, top_k=3):
    query_vec = embed_texts([query]).tolist()
    results = collection.query(query_embeddings=query_vec, n_results=top_k)

    print(f"Query: {query}\n")
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(f"distance={dist:.4f}  -  {doc}")

chroma_top_k_search("How does a computer understand a photo?", top_k=3)

---
# Lab 20: Document Question Answering using RAG
A simple, manual Retrieval-Augmented Generation pipeline: retrieve the most relevant
chunk from a document, then ask the LLM to answer using only that retrieved context.

In [ ]:
product_faq_doc = """
CloudSync Pro is a file synchronization service for teams. It offers real-time
file syncing across Windows, macOS, and Linux devices. The Free plan includes 5GB
of storage, while the Pro plan offers 1TB of storage for $9.99 per month. CloudSync
Pro supports two-factor authentication and end-to-end encryption for all files.
Team accounts can share folders with granular permission controls, allowing admins
to set read-only or edit access per member. Customer support is available 24/7
through live chat for Pro users, while Free users get email support with a
48-hour response time. Refunds are available within 14 days of a Pro subscription
purchase, no questions asked. The mobile app is available on both iOS and Android,
and supports offline access to previously synced files.
"""

# Split the document into small chunks (simple sentence-based chunking)
chunks = [c.strip() for c in product_faq_doc.split(".") if c.strip()]
print(f"Document split into {len(chunks)} chunks.")

In [ ]:
def rag_answer(question, chunks, top_k=2):
    chunk_embeddings = embed_texts(chunks)
    question_embedding = embed_texts([question])
    scores = cosine_similarity(question_embedding, chunk_embeddings)[0]

    top_indices = np.argsort(scores)[::-1][:top_k]
    retrieved_context = " ".join([chunks[i] for i in top_indices])

    prompt = f"""Answer the question using ONLY the context below. If the answer
is not in the context, say "I don't have that information."

Context: {retrieved_context}

Question: {question}
Answer:"""

    answer = ask_llm(prompt, max_tokens=150)
    return answer, retrieved_context

question = "How much storage does the Pro plan offer and what does it cost?"
answer, context_used = rag_answer(question, chunks)

print("Retrieved context:\n", context_used)
print("\nAnswer:\n", answer)

---
# Lab 21: End-to-End RAG Pipeline with LangChain
Document loading → text chunking → embeddings → retrieval → answer generation,
all wired together using LangChain.

In [ ]:
# 1. Document loading: save our sample document to a file and load it
with open("product_faq.txt", "w") as f:
    f.write(product_faq_doc)

from langchain_community.document_loaders import TextLoader

loader = TextLoader("product_faq.txt")
raw_documents = loader.load()
print("Document loaded. Number of raw documents:", len(raw_documents))

In [ ]:
# 2. Text chunking
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
split_docs = splitter.split_documents(raw_documents)
print("Document split into", len(split_docs), "chunks.")

In [ ]:
# 3. Embeddings + Vector store (Chroma)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

lc_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=lc_embeddings,
    collection_name="rag_pipeline_demo"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Vector store built and retriever ready.")

In [ ]:
# 4. Retrieval + Answer generation using a RetrievalQA chain
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

chat_llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name=MODEL_NAME)

qa_chain = RetrievalQA.from_chain_type(
    llm=chat_llm,
    retriever=retriever,
    return_source_documents=True
)

result = qa_chain.invoke({"query": "Does CloudSync Pro support offline access on mobile?"})

print("Answer:\n", result["result"])
print("\nSource chunks used:")
for doc in result["source_documents"]:
    print("-", doc.page_content)

---
# Lab 22: Domain-specific Chatbot using LangChain and a Vector Database
Building a chatbot restricted to a single domain — here, an **HR Policy Assistant**
that only answers using the company's HR policy document.

In [ ]:
hr_policy_doc = """
Employees are entitled to 18 days of paid annual leave and 10 days of sick leave
per calendar year. Sick leave beyond 2 consecutive days requires a medical
certificate. Employees may work from home up to 2 days per week with manager
approval, and full remote work requires HR approval for a fixed period. New
employees complete a 90-day probation period, during which either party may end
employment with 2 weeks' notice. After probation, the standard notice period is
30 days. Employees are eligible for health insurance from day one of employment,
covering the employee and up to 3 dependents. Overtime is compensated at 1.5x the
hourly rate for hours worked beyond 45 hours per week.
"""

with open("hr_policy.txt", "w") as f:
    f.write(hr_policy_doc)

hr_raw_docs = TextLoader("hr_policy.txt").load()
hr_chunks = splitter.split_documents(hr_raw_docs)

hr_vectorstore = Chroma.from_documents(
    documents=hr_chunks,
    embedding=lc_embeddings,
    collection_name="hr_policy_bot"
)
hr_retriever = hr_vectorstore.as_retriever(search_kwargs={"k": 2})

hr_chatbot = RetrievalQA.from_chain_type(llm=chat_llm, retriever=hr_retriever)
print("HR Policy chatbot ready!")

In [ ]:
def ask_hr_bot(question):
    response = hr_chatbot.invoke({"query": question})
    return response["result"]

print("Q: How many sick leave days do I get?")
print("A:", ask_hr_bot("How many sick leave days do I get?"))

print("\nQ: Can I work remotely full time?")
print("A:", ask_hr_bot("Can I work remotely full time?"))

---
# Lab 23: Context-aware Chatbot using LangChain, Retrieval, and an LLM
This chatbot remembers the conversation history, so it can understand follow-up
questions like "what about X?" that depend on earlier turns.

In [ ]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

conversational_chatbot = ConversationalRetrievalChain.from_llm(
    llm=chat_llm,
    retriever=hr_retriever,
    memory=memory
)
print("Context-aware chatbot ready!")

In [ ]:
def chat(question):
    response = conversational_chatbot.invoke({"question": question})
    print(f"You: {question}")
    print(f"Bot: {response['answer']}\n")

# A multi-turn conversation - notice the second question has no explicit subject
chat("What is the annual leave policy?")
chat("What about sick leave, how many days?")
chat("Does it need any proof if I take more than 2 days?")

---
# Lab 24: Simple AI Assistant that Answers Questions using an External Document
A lightweight assistant (no LangChain, built from scratch) that answers questions
grounded in one external document — useful for understanding what's happening
"under the hood" of a RAG system.

In [ ]:
onboarding_doc = """
Welcome to the company! On your first day, please report to the HR desk by 9:30 AM
to collect your laptop and ID card. Your manager will walk you through your team's
current projects during a 1-hour orientation meeting. IT will set up your email
and internal tools within the first 24 hours. All new employees must complete
mandatory compliance training within the first 2 weeks. Your first payroll cycle
begins the following month, and your salary is credited on the last working day
of each month. Please reach out to hr@company.com for any onboarding questions.
"""

onboarding_chunks = [c.strip() for c in onboarding_doc.split(".") if c.strip()]

def ai_assistant(question):
    answer, _ = rag_answer(question, onboarding_chunks, top_k=2)
    return answer

print("Q: When is my salary credited?")
print("A:", ai_assistant("When is my salary credited?"))

print("\nQ: What should I do on my first day?")
print("A:", ai_assistant("What should I do on my first day?"))

---
# Lab 25: Document-based AI Assistant Supporting Multiple Documents
Combining several documents into one vector store (with source tracking), so the
assistant can answer questions across all of them and cite which document it used.

*(Optional: if running in Google Colab, you can uncomment the upload cell below to
use your own files instead of the sample documents.)*

In [ ]:
# OPTIONAL: Upload your own .txt files in Google Colab (uncomment to use)

# from google.colab import files
# uploaded = files.upload()
# custom_file_names = list(uploaded.keys())

In [ ]:
from langchain.schema import Document as LC_Document

# Combine multiple documents, each tagged with a "source" in its metadata
all_documents = {
    "product_faq.txt": product_faq_doc,
    "hr_policy.txt": hr_policy_doc,
    "onboarding_guide.txt": onboarding_doc,
}

multi_docs = []
for source_name, text in all_documents.items():
    multi_docs.append(LC_Document(page_content=text, metadata={"source": source_name}))

multi_split_docs = splitter.split_documents(multi_docs)
print(f"Loaded {len(all_documents)} documents, split into {len(multi_split_docs)} chunks.")

In [ ]:
multi_vectorstore = Chroma.from_documents(
    documents=multi_split_docs,
    embedding=lc_embeddings,
    collection_name="multi_doc_assistant"
)
multi_retriever = multi_vectorstore.as_retriever(search_kwargs={"k": 3})

multi_doc_qa = RetrievalQA.from_chain_type(
    llm=chat_llm,
    retriever=multi_retriever,
    return_source_documents=True
)

def multi_doc_assistant(question):
    result = multi_doc_qa.invoke({"query": question})
    sources = sorted(set(doc.metadata["source"] for doc in result["source_documents"]))
    print("Answer:", result["result"])
    print("Sources used:", ", ".join(sources))

print("Multi-document AI assistant ready!\n")

In [ ]:
# Ask questions that span different documents
multi_doc_assistant("How much does the Pro plan cost?")
print()
multi_doc_assistant("How many annual leave days do employees get?")
print()
multi_doc_assistant("What happens on an employee's first day?")

---
## End of Notebook
Labs 16 to 25 are complete:
- Lab 16-17: Embeddings and semantic search with cosine similarity.
- Lab 18-19: Vector databases with FAISS and ChromaDB.
- Lab 20-21: Manual RAG and a full LangChain RAG pipeline.
- Lab 22-23: Domain-specific and context-aware chatbots using LangChain.
- Lab 24-25: Simple and multi-document AI assistants for Q&A over external documents.
